# Notebook Overview

This notebook is used to investigate rules to improve the NER model prediction by removing erroneous predictions in a post-processing step. The notebook assumes that the model directory created using the Hugging Face API is linked in the current directory as ./model.

In [1]:
"""#! pip install transformers
#! pip install torch"""
print("Passed")

Passed


In [2]:
import transformers
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# setup named entity recognizer using pre-trained model
model_path = './bert-finetuned-ner/checkpoint-200'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path, ignore_mismatched_sizes=True)
nlp = pipeline('ner', model=model, tokenizer=tokenizer, aggregation_strategy='first')

/opt/homebrew/anaconda3/envs/privacy_mac/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use mps:0


In [3]:
import json

# read and merge the two scenario datasets
dataset = json.load(open('scenarios-training-0.json'))
scenarios = dataset['test']

In [4]:
print(len(scenarios))

30


In [8]:
import nltk
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sungyuuli/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/sungyuuli/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [5]:
# extract entities from the scenario text (long running process)
entities = []
for scenario in scenarios:
    results = nlp(scenario['text'])
    for r in results:
        r['text'] = scenario['text']
        entities.append(r)

In [6]:
print('Found %i entities across %i scenarios.' % (len(entities), len(scenarios)))

Found 594 entities across 30 scenarios.


In [9]:
import nltk

# returns the start and end word indices for the phrase in words
def word_index(words, phrase, start_char):
    # correct certain parse errors
    extras = ["' ", " - ", " / "]
    for i in range(len(extras)):
        j = phrase.find(extras[i])
        if j >= 0:
            phrase = phrase[:j] + extras[i].strip() + phrase[j+len(extras[i]):]
        
    # using the start char, find the first and last word index for phrase
    char_index = 0
    phrase_len = len(phrase.split())
    for i in range(len(words)):
        char_index += len(words[i])
        if char_index + i >= start_char:
            j = i
            while i < len(words):
                if i + phrase_len > len(words):
                    return None
                elif ' '.join(words[i:i + phrase_len]) == phrase:
                    return (i, i + phrase_len)
                i += 1
    return None

# construct word, tag lists for phrase and words to the left and right of phrase
errors = 0
parsed = {}
for e in entities:
    # reuse the tagged scenario texts
    if not e['text'] in parsed:
        tags = nltk.pos_tag(nltk.word_tokenize(e['text']))
        parsed[e['text']] = tags
    tags = parsed[e['text']]
    
    # create separate word list and use to estimate word indices from char indices
    words = [w for (w, p) in tags]
    index = word_index(words, e['word'], e['start'])
    
    # save the associated word, tag lists
    e['p_words'] = nltk.pos_tag(nltk.word_tokenize(e['text'][e['start']:e['end']]))
    if not index:
        e['l_words'] = []
        e['r_words'] = []
        errors += 1
    else:
        e['r_words'] = tags[index[1]:index[1] + 3]
        e['l_words'] = tags[index[0] - 3:index[1]]
        
# report any phrases that could not be indexed
print('Unable to find word %i/%i boundaries.' % (errors, len(entities)))

Unable to find word 7/594 boundaries.


In [10]:
import nltk

# filter out incomplete phrases based on a few simple rules
filtered = []
excluded = []
for e in entities:
    pos = e['p_words']
    # remove phrases ending in 'the', 'a', 'and', 'or,' or 'your', for example
    if pos[-1][1].startswith('DT') or pos[-1][1].startswith('CC') or pos[-1][1].startswith('PRP$'):
        excluded.append((1, e['word'], pos))
    # remove phrases beginning with POS
    elif pos[0][1] == 'POS' or pos[0][0] == 'of' or pos[0][1] == 'CC':
        excluded.append((3, e['word'], pos))
    # remove phrases less than two words not ending in NN or VBG
    elif len(pos) == 1 and not pos[0][1].startswith('NN') and not pos[0][1].startswith('VBG'):
        excluded.append((2, e['word'], pos))
    elif len(pos) == 2 and not pos[0][1].startswith('NN') and not pos[1][1].startswith('NN') and not pos[1][1].startswith('VBG'):
        excluded.append((2, e['word'], pos))
    else:
        filtered.append(e)

print('Filtered from %i to %i unique entities' % (len(entities), len(filtered)))
print('Excluded %i entites' % len(excluded))
print('Enhanced precision: %0.2f' % ((len(entities) - ((1.0 - 0.66) * len(entities))) / len(filtered)))

Filtered from 594 to 563 unique entities
Excluded 31 entites
Enhanced precision: 0.70


In [12]:
print(f"Results 内容样例: {results[:5]}") # 打印前5个看看
print(f"Results 类型: {type(results[0]) if results else '空列表'}")

Results 内容样例: [{'entity_group': 'SIM', 'score': 0.77489394, 'word': 'games', 'start': 32, 'end': 37, 'text': 'It keeps the data of the future games I want to play on steam. Even games that are not released yet. Thus, steam gets a better idea of what to recommend to me in terms of games on the main page when I sign in. This will improve the chances that I end up buying a game on the recommended screen rather than not purchasing anything. For example, on the screenshot you can see that I have a game called "Undisputed" set to be released next year. The game is a boxing video game in the "sports" genre. Steam might then get the idea that I am into sports games and market that towards me in the future and show me other sports games on my recommended screen that I then may feel inclined to purchase because they see that I am the type of gamer that loves to play sports games.', 'p_words': [('games', 'NNS')], 'r_words': [('I', 'PRP'), ('want', 'VBP'), ('to', 'TO')], 'l_words': [('of', 'IN'), 

In [13]:
import csv

# sort phrases alphabetically
results.sort(key=lambda x:x.get('word', ''))  # 使用 get 方法避免 KeyError

# write phrases to a file
with open('information_types.csv', 'w') as f:
    writer = csv.writer(f)
    for row in results:
        writer.writerow(row)